In [1]:
import numpy as np

In [ ]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, "..")
from importlib import reload

import equations as eq
import trajectory_lib as tr
reload (tr);
reload (eq);

# motion_list  =  ['Flexion_yzy','Elevation_yzy','Scabduction_yzy'] #
# participant = ['1LU','8tata','11NI']
participant = ['3NA']
motion_list  =  ['Elevation_yzy'] #
GH_seq = 'YZY' 
weight= 50

# OS_struct = sc.io.loadmat('../Motions/'+participant+'/Elevation_yzy/OS_model.mat')
include_activation_dynamics = True
act_w = 1
interval_value = 0.04

w_diff_exc = 5e-2
w_diff_vel = 5e-2

In [ ]:
for ipar in range(len(participant)):
    for imot in range(len(motion_list)):
        OS_struct = sc.io.loadmat('../Motions/'+participant[ipar]+'/OS_model.mat')
        # MM,FO,q,u,fr,frstar,kindeq,xdot,first_elips_scale,elips_trans = eq.create_eoms_eul(OS_struct,derive = 'numeric',gen_matlab_functions = 0,GH_seq = GH_seq)
        # TE,activations,TE_conoid = eq.polynomials_euler(OS_struct,q,u,derive = 'numeric')

        traj_w = weight
        struct_name = 'res_euler_'+motion_list[imot]+'_'+str(weight)
        eoms_implicit = sp.Matrix(kindeq).col_join(fr+frstar+TE+sp.Matrix(TE_conoid))

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            # print(activations)
            for i in range(len(activations)):
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))

            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)
        reload(eq);
        # num_nodes = 101
        file = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/'+motion_list[imot]

        
        traj_original, num_nodes, time = tr.exp_trajectory_eul(file,interval_value)
        traj = tr.exp_trajectory_eul_myobj(traj_original,GH_seq)
        q0_t0 = traj_original[:,0][:3]

        if include_activation_dynamics:
            state_symbols = tuple(q+u+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols)
        num_q = len(q)
        num_u = len(u)
        num_faux = 0

        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        objective_traj,objective_traj_jac, objective_traj_t0, objective_traj_t0_jac = eq.custom_objective_eul(len(q),interval_value, GH_seq = GH_seq)
        obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes,interval_value)
        w_diff_act = 0
        

        def obj(free):
            min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
            min_SC_t0 = traj_w * np.sum(objective_traj_t0(free[0::num_nodes][:3],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
            min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)

            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))))

            obj = (min_traj + min_torque + min_vel_dif + min_SC_t0) # 

            if include_activation_dynamics:
                obj += min_exc_dif
                
            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            grad[:num_q*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))
            grad[0::num_nodes][:3] += traj_w * np.sum(objective_traj_t0_jac(free[0::num_nodes][:3],q0_t0))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] += act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] #+ w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
            if include_activation_dynamics:
                grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))

            return grad
        
        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01)))
        instance_constraints = []
        instance_constraints.append(state_symbols[-1].func(0.0)-0)

        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)

        for i in range(num_q):
            if i == 6 or i == 8:
                bndrs.update({q[i]: (min(traj_original[i,:])-1, max(traj_original[i,:])+1)})
            else:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.2, max(traj_original[i,:])+0.2)})

        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
        ) #               
        time_to_create = tm.time() - start
        print(time_to_create)
        prob.add_option('max_iter',1000)
        prob.add_option('limited_memory_max_history', 40)
        reload(tr);
        initial_guess = np.zeros(prob.num_free)
        initial_guess[:10*num_nodes] = traj_original.flatten()


        time_2_solve_start = tm.time()
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)
        reload(tr)

        file_name = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mat'
        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mot'
        tr.sol2mot_eul(solution, num_nodes, len(q), time, file_name_mot,GH_seq)

79
obj_check 520.5816418584325
obj_grad_check -1624.2954841507626
1331.8780148029327
This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  8055601
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    27674
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    26664
                     variables with only upper bounds:        0
Total number of equality constraints.................:    14701
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha

In [17]:
w_diff_exc = 1e-3
w_diff_vel = 1e-3
traj_w = 50
prob.add_option('max_iter',1500)
time_2_solve_start = tm.time()
solution, info = prob.solve(initial_guess)
time_2_solve = tm.time() - time_2_solve_start
print(info['status_msg'])
print(info['obj_val'])
act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)


file_name = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mat'
tr.sol2struct(solution,activations,num_q,num_u,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

file_name_mot = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mot'
tr.sol2mot_eul(solution, num_nodes, len(q), time, file_name_mot,GH_seq)

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  8055601
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    27674
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    26664
                     variables with only upper bounds:        0
Total number of equality constraints.................:    14701
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.1307897e-02 4.02e+02 1.51e-03   0.0 0.00e+00    -  0.00e+00 0.00e+00 